# Rubik's Cube Transformer: InfoNCE Contrastive Training on 2x NVIDIA T4 GPUs

This notebook trains a **Rubik's Cube Transformer Encoder** using large-scale **InfoNCE contrastive learning** on **2x NVIDIA T4 GPUs** using PyTorch Distributed Data Parallel (`torchrun` DDP).

### Topology & Contrastive Rules:
- **Positive Pair**: Exactly 1 move ahead ($\text{dist} = 1$).
- **Excluded Buffer Zone**: States with $2 \le \text{dist} \le 5$ are strictly excluded ($k=5$).
- **Negative Pairs**: Sampled states strictly with $\text{dist} > 5$ ($K=100$ negatives per anchor).
- **Streaming Generator**: Infinite on-the-fly random generation directly in RAM (zero dataset files on disk, no epoch cycling).
- **Hardware**: Dual NVIDIA T4 GPUs with Automatic Mixed Precision (AMP FP16) and parallel CPU DataLoader workers.


## 1. Hardware & Multi-GPU Detection
Verify that Kaggle has allocated **2x NVIDIA T4 GPUs** (`cuda:0` and `cuda:1`).

In [ ]:
!nvidia-smi

import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
device_count = torch.cuda.device_count()
print(f"GPU Count:       {device_count}")
for i in range(device_count):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | VRAM: {props.total_memory / (1024**3):.2f} GB")

if device_count < 2:
    print("\n⚠️ Notice: Less than 2 GPUs detected. In Kaggle Notebook Settings, select 'Accelerator' -> 'GPU T4 x 2'.")
else:
    print("\n✅ Dual NVIDIA T4 GPUs successfully detected and ready for DDP training!")

## 2. Setup Repository & Dependencies
Clone the `rubik-ml-solver` repository from GitHub and install it in editable mode.

In [ ]:
import os

# Clone repository if not already present
if not os.path.exists("rubik"):
    !git clone https://github.com/luisportillo91/rubik-ml-solver.git
    %cd rubik-ml-solver
else:
    print("Working inside rubik-ml-solver repository.")

# Install package dependencies
!pip install -q -e .
!pip install -q matplotlib pytest

## 3. Verify Codebase via Test Suite
Execute the complete unit test suite to verify cube permutation logic, bidirectional BFS search, transformer encoder, and InfoNCE loss.

In [ ]:
!pytest -v tests/

## 4. Multi-GPU InfoNCE Training (2x T4 GPUs via DDP)

We launch `scripts/train_infonce.py` via `torchrun --nproc_per_node=2`:
- **2 Processes**: Each bound to one NVIDIA T4 (`cuda:0` and `cuda:1`).
- **Batch Size**: 32 per GPU $\rightarrow$ Global Batch = 64 anchors per step.
- **Negatives**: 100 per anchor $\rightarrow$ $64 \times 100 = 6,400$ verified negatives evaluated per step.
- **Distance Buffer**: $k=5$ excluded zone ($2 \le \text{dist} \le 5$).
- **CPU Workers**: 2 DataLoader background worker processes per GPU (4 workers total) streaming scrambles asynchronously in RAM.
- **AMP**: FP16 Automatic Mixed Precision utilizing NVIDIA Turing Tensor Cores.
- **Checkpoints**: Saved to `/kaggle/working/checkpoints` for permanent storage in Kaggle outputs.

In [ ]:
import os

save_dir = "/kaggle/working/checkpoints"
os.makedirs(save_dir, exist_ok=True)

# Number of steps (adjust according to your time budget)
STEPS = 2000
BATCH_SIZE_PER_GPU = 32
K_NEGATIVES = 100
DISTANCE_THRESHOLD = 5
NUM_WORKERS = 1

device_count = torch.cuda.device_count()
nproc = max(1, device_count)

print(f"Launching training on {nproc} GPU(s) with {STEPS} total steps...")

!torchrun --nproc_per_node={nproc} scripts/train_infonce.py \
    --steps {STEPS} \
    --batch-size {BATCH_SIZE_PER_GPU} \
    --k-negatives {K_NEGATIVES} \
    --distance-threshold {DISTANCE_THRESHOLD} \
    --d-model 128 \
    --nhead 4 \
    --num-layers 4 \
    --d-embed 128 \
    --lr 3e-4 \
    --log-every 25 \
    --save-every 250 \
    --num-workers {NUM_WORKERS} \
    --save-dir {save_dir} \
    --history-file training_history.json

## 5. Visualize Training Curves
Plot the evolution of InfoNCE loss, Top-1 / Top-5 retrieval accuracy, positive vs negative cosine similarity, and similarity margin across optimization steps.

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = os.path.join(save_dir, "training_history.json")
if os.path.exists(history_path):
    with open(history_path, "r") as f:
        history = json.load(f)

    steps = [h["step"] for h in history]
    losses = [h["loss"] for h in history]
    top1 = [h["top1_acc"] for h in history]
    top5 = [h["top5_acc"] for h in history]
    pos_sim = [h["pos_sim"] for h in history]
    neg_sim = [h["neg_sim"] for h in history]
    margins = [h["margin"] for h in history]
    speeds = [h.get("steps_per_sec", 0) for h in history]

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle("Rubik's Cube Transformer InfoNCE Training (2x T4 GPUs, K=100 Negatives)", fontsize=16, fontweight="bold")

    # 1. Contrastive Loss
    axes[0, 0].plot(steps, losses, color="#1f77b4", lw=2)
    axes[0, 0].set_title("InfoNCE Contrastive Loss", fontweight="bold")
    axes[0, 0].set_xlabel("Step")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].grid(True, alpha=0.3)

    # 2. Retrieval Accuracy
    axes[0, 1].plot(steps, top1, label="Top-1 Acc (%)", color="#2ca02c", lw=2)
    axes[0, 1].plot(steps, top5, label="Top-5 Acc (%)", color="#ff7f0e", lw=2)
    axes[0, 1].set_title("1-Step Move Retrieval Accuracy (1 out of 101)", fontweight="bold")
    axes[0, 1].set_xlabel("Step")
    axes[0, 1].set_ylabel("Accuracy (%)")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # 3. Cosine Similarities
    axes[1, 0].plot(steps, pos_sim, label="Positive (dist=1)", color="#2ca02c", lw=2)
    axes[1, 0].plot(steps, neg_sim, label="Negatives (dist>5)", color="#d62728", lw=2)
    axes[1, 0].set_title("Positive vs Negative Cosine Similarity", fontweight="bold")
    axes[1, 0].set_xlabel("Step")
    axes[1, 0].set_ylabel("Cosine Similarity")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # 4. Margin & Speed
    axes[1, 1].plot(steps, margins, color="#9467bd", lw=2, label="Margin (Pos - Neg)")
    axes[1, 1].set_title("Representation Margin", fontweight="bold")
    axes[1, 1].set_xlabel("Step")
    axes[1, 1].set_ylabel("Margin")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    curve_plot_path = "/kaggle/working/training_curves.png"
    plt.savefig(curve_plot_path, dpi=150)
    print(f"Training curve plot saved to: {curve_plot_path}")
    plt.show()
else:
    print(f"History file not found at {history_path}")

## 6. Learned Distance Manifold Evaluation
Verify whether the trained transformer encoder has learned a monotonic distance metric on the Rubik's cube graph.
We measure cosine similarity between cube states as a function of scramble distance $d \in [1, 20]$.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from rubik.cube import create_solved_cube, scramble
from rubik.encoder import CubeTransformerEncoder, batch_to_tensor

best_checkpoint = os.path.join(save_dir, "best_encoder.pt")
ckpt = torch.load(best_checkpoint, map_location="cpu")
saved_args = ckpt.get("args", {})

eval_model = CubeTransformerEncoder(
    d_model=saved_args.get("d_model", 128),
    nhead=saved_args.get("nhead", 4),
    num_layers=saved_args.get("num_layers", 4),
    dim_feedforward=saved_args.get("d_model", 128) * 2,
    d_embed=saved_args.get("d_embed", 128),
    pooling="cls",
)
eval_model.load_state_dict(ckpt["model_state_dict"])
eval_model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
eval_model.to(device)

distances = list(range(1, 21))
trials_per_distance = 64
mean_sims = []
std_sims = []

solved = create_solved_cube()

with torch.no_grad():
    for d in distances:
        sims = []
        for _ in range(trials_per_distance):
            s1, _ = scramble(solved, n_moves=12)
            s2, _ = scramble(s1, n_moves=d)
            
            t1 = batch_to_tensor([s1], device=device)
            t2 = batch_to_tensor([s2], device=device)
            
            e1, _ = eval_model(t1)
            e2, _ = eval_model(t2)
            
            e1_norm = F.normalize(e1, p=2, dim=-1)
            e2_norm = F.normalize(e2, p=2, dim=-1)
            sim = (e1_norm * e2_norm).sum(dim=-1).item()
            sims.append(sim)
        mean_sims.append(float(np.mean(sims)))
        std_sims.append(float(np.std(sims)))

plt.figure(figsize=(10, 5))
plt.plot(distances, mean_sims, marker='o', color='#1f77b4', lw=2, label='Average Cosine Similarity')
plt.fill_between(
    distances,
    [m - s for m, s in zip(mean_sims, std_sims)],
    [m + s for m, s in zip(mean_sims, std_sims)],
    alpha=0.2,
    color='#1f77b4',
)
plt.axvline(x=1, color='green', linestyle='--', label='Positive (dist=1)')
plt.axvspan(2, 5, color='orange', alpha=0.2, label='Excluded Zone (k=5)')
plt.axvline(x=5.5, color='red', linestyle='--', label='Negative Zone (dist>5)')
plt.title("Learned Representation Similarity vs Scramble Distance", fontsize=14, fontweight="bold")
plt.xlabel("Scramble Distance (moves)", fontsize=12)
plt.ylabel("Cosine Similarity", fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
manifold_plot_path = "/kaggle/working/similarity_vs_distance.png"
plt.savefig(manifold_plot_path, dpi=150)
print(f"Distance manifold plot saved to: {manifold_plot_path}")
plt.show()

## 7. Output Artifacts
List all saved model checkpoints and visualization plots in `/kaggle/working/` ready for download from the Kaggle sidebar.

In [ ]:
!ls -lh /kaggle/working/checkpoints/
!ls -lh /kaggle/working/*.png
print("\n🎉 Training and evaluation finished successfully!")